# 📦 RepoTopology: Package & Validate Kaggle Submission in Google Colab
### *Google - The Gemma 4 Developer Agent Competition*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/catlikeflyer/repotopology/blob/main/notebooks/03_package_submission.ipynb)

This notebook verifies the declarative agent submission schema (`adk-submission` / `swegemma`), attaches your trained LoRA adapter from Google Drive, and packages `submission.zip` ready for one-click download.

## 1. Setup & Mount Google Drive

In [ ]:
import os
import shutil
import zipfile
from pathlib import Path
from google.colab import files

# If running inside Colab and repo not present, clone it
if not os.path.exists("repotopology") and not os.path.exists("pyproject.toml"):
    !git clone https://github.com/catlikeflyer/repotopology.git repotopology
    %cd repotopology

# Mount Drive to retrieve trained LoRA weights
try:
    from google.colab import drive
    drive.mount('/content/drive')
    has_drive = True
except Exception as e:
    has_drive = False
    print(f"Drive mount skipped: {e}")

## 2. Retrieve & Place LoRA Adapter

In [ ]:
adapter_target_dir = Path("./submission/adapters/main_lora")
adapter_target_dir.mkdir(parents=True, exist_ok=True)

# Look for trained adapter zip in Google Drive or local path
drive_zip = Path("/content/drive/MyDrive/main_lora_31b.zip")
local_zip = Path("/content/main_lora_31b.zip")

if drive_zip.exists():
    print(f"Found adapter in Google Drive: {drive_zip}")
    shutil.unpack_archive(str(drive_zip), "/tmp/adapter_extracted")
    # Copy adapter files into submission/adapters/main_lora/
    extracted_source = Path("/tmp/adapter_extracted/adapters/main_lora")
    if not extracted_source.exists():
        extracted_source = Path("/tmp/adapter_extracted")
    for item in extracted_source.iterdir():
        if item.is_file():
            shutil.copy2(item, adapter_target_dir / item.name)
    print(f"[SUCCESS] Extracted adapter files to {adapter_target_dir}")
elif local_zip.exists():
    shutil.unpack_archive(str(local_zip), "/tmp/adapter_extracted")
    extracted_source = Path("/tmp/adapter_extracted/adapters/main_lora")
    if not extracted_source.exists():
        extracted_source = Path("/tmp/adapter_extracted")
    for item in extracted_source.iterdir():
        if item.is_file():
            shutil.copy2(item, adapter_target_dir / item.name)
    print(f"[SUCCESS] Extracted adapter files from local session zip.")
else:
    print("[NOTICE] No pre-trained adapter zip found. Using declarative zero-shot agent configuration.")

## 3. Validate Declarative Submission Schema

In [ ]:
import yaml

# Validate agent.yaml
agent_yaml_path = Path("./submission/agent.yaml")
assert agent_yaml_path.exists(), "submission/agent.yaml is missing!"
with open(agent_yaml_path, "r", encoding="utf-8") as f:
    agent_raw = f.read()
print("--- submission/agent.yaml ---")
print(agent_raw)

# Validate prompts
assert Path("./submission/prompts/system.md").exists(), "Missing prompts/system.md!"
assert Path("./submission/prompts/analyzer.md").exists(), "Missing prompts/analyzer.md!"
print("[PASS] All declarative prompt templates verified.")

## 4. Package `submission.zip` & Download

In [ ]:
!pip install -q -e "."
from repotopology.package_submission import create_submission_bundle

zip_path = create_submission_bundle(
    submission_dir="./submission",
    output_zip="./submission.zip",
    model_name="gemma-4-e4b-it-qat-w4a16-ct",
)

# Inspect created zip archive
with zipfile.ZipFile(zip_path, "r") as z:
    print("\n--- Archive Contents ---")
    for info in z.infolist():
        print(f"{info.filename:45s} {info.file_size:>8d} bytes")

# Save copy to Google Drive
if has_drive:
    drive_sub = Path("/content/drive/MyDrive/submission.zip")
    shutil.copy2(zip_path, drive_sub)
    print(f"\n[SUCCESS] Backed up to Google Drive at: {drive_sub}")

# Download to local computer
print("\n[SUCCESS] Downloading submission.zip to your local computer...")
files.download(str(zip_path))